- YOLOv26 → Detection
- RESNET50 → Classification
- Gradio UI Demo

In [1]:
import os
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models

import gradio as gr

d:\yolov26\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DETECT_MODEL_PATH = r"D:\yolov26\pipeline1\weight1\best.pt"
CLASS_MODEL_PATH  = r"D:\yolov26\pipeline1\best_classification.pth"

CLASS_NAMES  = ["Glioma", "Meningioma", "Pituitary"]


In [3]:
from ultralytics import YOLO
yolo_model = YOLO(DETECT_MODEL_PATH)

In [4]:
def load_resnet50():
    model = models.resnet50(weights=None)
    
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    
    state = torch.load(CLASS_MODEL_PATH, map_location="cpu")
    model.load_state_dict(state, strict=False)
    
    model.eval()
    return model

resnet_model = load_resnet50()

C:\Users\Asus Rog Strix\AppData\Local\Temp\ipykernel_10316\430340811.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CLASS_MODEL_PATH, map_location="c

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [6]:
def detect_tumor(image):
    results = yolo_model.predict(image, conf=0.25, verbose=False)

    boxes = []
    has_positive = False

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = r.names[cls]  # "positive" hoặc "negative"
            conf = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            boxes.append({
                "label": label,
                "conf": conf,
                "bbox": (x1, y1, x2, y2)
            })

            if label == "positive":
                has_positive = True

    return boxes, has_positive

In [7]:
def classify_full_image(image):
    img = Image.fromarray(image).convert("RGB")
    t = transform(img).unsqueeze(0)

    with torch.no_grad():
        out = resnet_model(t)
        probs = torch.softmax(out, dim=1)[0].numpy()

    idx = int(np.argmax(probs))
    return CLASS_NAMES[idx], float(probs[idx])

In [8]:
def draw_boxes(image, boxes):
    img = image.copy()

    for b in boxes:
        x1, y1, x2, y2 = b["bbox"]
        label = b["label"]
        conf = b["conf"]

        color = (0,255,0) if label == "negative" else (255,0,0)

        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img, f"{label} {conf:.2f}",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color, 2)

    return img

In [9]:
def pipeline(image):
    if image is None:
        return None, "Vui lòng upload ảnh"

    # Step 1: Detection
    boxes, has_positive = detect_tumor(image)
    img_draw = draw_boxes(image, boxes)
    
    # Step 3: Classification trên FULL IMAGE
    cls, conf = classify_full_image(image)

    result_text = f"""
    ⚠️ Phát hiện khối u

    👉 Loại u: {cls}
    """

    return img_draw, result_text

In [10]:
demo = gr.Interface(
    fn=pipeline,
    inputs=gr.Image(type="numpy"),
    outputs=[
        gr.Image(label="Detection Result"),
        gr.Textbox(label="Kết quả")
    ],
    title="Brain Tumor Detection + Classification (No Crop)"
)

In [11]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
